# 🚀 AutoPilot — Autonomous YouTube Video Factory
**Keep PRIVATE** | GPU: T4 required

### Cells
| Cell | What it does | Run once? |
|---|---|---|
| 1 | Install + clone + patch for Kaggle | Per session |
| 2 | Write API keys to .env | Per session |
| 3 | Run pipeline (auto-picks trending topic) | Every video |
| 4 | View + download generated videos | Anytime |
| 5 | Batch mode (multiple niches overnight) | Optional |

### API Keys
**Recommended:** Use Kaggle Secrets (Add-ons > Secrets) with these labels:
- `GROQ_API_KEYS` — comma-separated GROQ keys
- `GEMINI_API_KEYS` — comma-separated Gemini keys
- `PEXELS_API_KEYS` — Pexels B-roll key
- `DEEPSEEK_API_KEYS`, `NVIDIA_API_KEY`, `OPENAI_API_KEY` (optional)

Or paste keys directly in Cell 2 where it says `YOUR_xxx_HERE`.

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1 — INSTALL + CLONE  (run once per session)
# ═══════════════════════════════════════════════════════════
import subprocess, sys, os

CLONE_DIR    = '/kaggle/working/autopilot'
PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'

# ── 1. System packages ─────────────────────────────────────
print('[1/4] System packages...')
subprocess.run(['apt-get', 'install', '-y', '-q', 'ffmpeg', 'libsndfile1'],
               capture_output=True)
r = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
print('      ffmpeg:', r.stdout.split('\n')[0])

# ── 2. Python packages ─────────────────────────────────────
print('[2/4] Python packages...')
pkgs = [
    'structlog',
    'python-dotenv',
    'edge-tts',
    'nest_asyncio',
    'openai',
    'groq',
    'requests',
    'pillow',
    'pydantic>=2.0',
    'httpx',
]
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + pkgs,
    capture_output=True, text=True
)
if r.returncode != 0:
    print('  ERROR:', r.stderr[-300:])
else:
    print('  All packages OK')

# ── 3. Clone / pull latest pipeline code ───────────────────
print('[3/4] Latest pipeline from GitHub...')
if os.path.exists(CLONE_DIR):
    r = subprocess.run(['git', '-C', CLONE_DIR, 'pull'],
                       capture_output=True, text=True)
    print('     ', r.stdout.strip() or 'Already up to date')
else:
    r = subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/rajatsarswat2001/autopilot.git', CLONE_DIR],
        capture_output=True, text=True
    )
    print('     ', 'Cloned OK' if r.returncode == 0 else 'FAILED: ' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError('Clone failed')

for d in ['outputs/video', 'outputs/audio', 'outputs/visual', 'data/clip_cache']:
    os.makedirs(os.path.join(PIPELINE_DIR, d), exist_ok=True)

# ── 4. Apply nest_asyncio (Edge TTS needs this in Jupyter) ─
print('[4/4] Applying nest_asyncio...')
import nest_asyncio
nest_asyncio.apply()

import importlib, torch
for pkg in ['langchain', 'langgraph', 'structlog', 'edge_tts']:
    try:
        m = importlib.import_module(pkg)
        v = getattr(m, '__version__', '?')
        print(f'  {pkg:<20} {v}')
    except Exception as e:
        print(f'  {pkg:<20} MISSING - {e}')
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'
print(f'  GPU                 {gpu}')

print('=' * 50)
print('Setup complete - run Cell 2 to add API keys')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2 — API KEYS  (run once per session)
# ═══════════════════════════════════════════════════════════
# Option A: Kaggle Secrets (recommended - set in Add-ons > Secrets)
# Option B: Paste your keys below where it says YOUR_xxx_HERE
import os

PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'

# ── Try Kaggle Secrets first ───────────────────────────────
keys = {}
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    secret_map = {
        'GROQ_API_KEYS':     'GROQ_API_KEYS',
        'GEMINI_API_KEYS':   'GEMINI_API_KEYS',
        'DEEPSEEK_API_KEYS': 'DEEPSEEK_API_KEYS',
        'NVIDIA_API_KEY':    'NVIDIA_API_KEY',
        'OPENAI_API_KEY':    'OPENAI_API_KEY',
        'PEXELS_API_KEYS':   'PEXELS_API_KEYS',
        'PIXABAY_API_KEYS':  'PIXABAY_API_KEYS',
        'TAVILY_API_KEY':    'TAVILY_API_KEY',
    }
    for env_key, secret_label in secret_map.items():
        try:
            val = _s.get_secret(secret_label)
            if val:
                keys[env_key] = val
                print(f'  OK {env_key} (from Kaggle Secrets)')
        except Exception:
            pass
except ImportError:
    print('  Kaggle Secrets not available - using hardcoded keys')

# ── Fallback: paste your keys below ────────────────────────
if 'GROQ_API_KEYS' not in keys:
    GROQ_KEYS = [
        'YOUR_GROQ_KEY_1',
        'YOUR_GROQ_KEY_2',
        # add more keys for rate-limit rotation
    ]
    keys['GROQ_API_KEYS'] = ','.join(GROQ_KEYS)
    print(f'  GROQ_API_KEYS ({len(GROQ_KEYS)} keys, hardcoded)')

if 'GEMINI_API_KEYS' not in keys:
    GEMINI_KEYS = [
        'YOUR_GEMINI_KEY_1',
        'YOUR_GEMINI_KEY_2',
        # add more keys for rate-limit rotation
    ]
    keys['GEMINI_API_KEYS'] = ','.join(GEMINI_KEYS)
    print(f'  GEMINI_API_KEYS ({len(GEMINI_KEYS)} keys, hardcoded)')

if 'DEEPSEEK_API_KEYS' not in keys:
    keys['DEEPSEEK_API_KEYS'] = 'YOUR_DEEPSEEK_KEY'
    print('  DEEPSEEK_API_KEYS (hardcoded)')

if 'PEXELS_API_KEYS' not in keys:
    keys['PEXELS_API_KEYS'] = 'YOUR_PEXELS_KEY'
    print('  PEXELS_API_KEYS (hardcoded)')

if 'PIXABAY_API_KEYS' not in keys:
    keys['PIXABAY_API_KEYS'] = 'YOUR_PIXABAY_KEY'
    print('  PIXABAY_API_KEYS (hardcoded)')

# ── Pipeline settings ──────────────────────────────────────
keys['AUTOPILOT_AUTO_APPROVE'] = '1'
keys['AUDIO_PARALLEL_WORKERS'] = '4'
keys['VISUAL_PARALLEL_WORKERS'] = '4'
keys['LOG_LEVEL'] = 'INFO'

env_path = os.path.join(PIPELINE_DIR, '.env')
with open(env_path, 'w') as f:
    for k, v in keys.items():
        f.write(f'{k}={v}\n')
        os.environ[k] = v

n_keys = sum(1 for k in keys if 'KEY' in k or 'TOKEN' in k)
print(f'\n.env written: {env_path}')
print(f'Total API keys configured: {n_keys}')
if any('YOUR_' in str(v) for v in keys.values()):
    print('\n!! WARNING: Some keys are still placeholders !!')
    print('   Paste your real keys above or use Kaggle Secrets')
print('\nRun Cell 3 to start generating a video')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3 — RUN PIPELINE  (generates one video ~10-20 min)
# ═══════════════════════════════════════════════════════════
import subprocess, sys, os, time
from pathlib import Path

PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'

# ── CONFIG ──────────────────────────────────────────────────
# personal_finance | saas_tools | legal_tax | senior_health | real_estate
NICHE = 'personal_finance'
TOPIC = ''   # leave empty = auto-detect trending topic

# ── Pre-run checks ─────────────────────────────────────────
checks = {
    'Pipeline dir': os.path.exists(PIPELINE_DIR),
    'main.py':      os.path.exists(os.path.join(PIPELINE_DIR, 'main.py')),
    '.env':         os.path.exists(os.path.join(PIPELINE_DIR, '.env')),
}
print('PRE-RUN CHECKS')
for k, v in checks.items():
    print(f'  {"OK" if v else "MISSING"} {k}')
    if not v:
        raise RuntimeError(f'{k} missing - run Cell 1 and 2 first')

import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None (CPU)'
print(f'  GPU: {gpu}')
print(f'  Niche: {NICHE}')
print(f'  Topic: {TOPIC or "auto-detect"}')
print('=' * 60)

cmd = [
    sys.executable, 'main.py',
    '--niche', NICHE,
    '--no-db',
    '--approve',
    '--log-format', 'console',
]
if TOPIC:
    cmd += ['--topic', TOPIC]

start = time.time()
proc = subprocess.Popen(
    cmd,
    cwd=PIPELINE_DIR,
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1
)
all_lines = []
for line in proc.stdout:
    print(line, end='', flush=True)
    all_lines.append(line)
proc.wait()

elapsed = time.time() - start
print('=' * 60)
print(f'EXIT CODE : {proc.returncode}')
print(f'DURATION  : {elapsed:.0f}s ({elapsed/60:.1f} min)')

if proc.returncode != 0:
    print()
    print('FAILED - last 30 lines (paste in chat to debug):')
    print(''.join(all_lines[-30:]))
else:
    print()
    print('SUCCESS!')
    vid_dir = Path(PIPELINE_DIR) / 'outputs' / 'video'
    videos = sorted(vid_dir.glob('*.mp4'), key=lambda x: x.stat().st_mtime, reverse=True)
    print(f'Videos generated: {len(videos)}')
    for v in videos[:5]:
        print(f'  {v.name}  ({v.stat().st_size/1024/1024:.1f} MB)')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4 — VIEW & DOWNLOAD RESULTS
# ═══════════════════════════════════════════════════════════
from pathlib import Path
from IPython.display import HTML, display

PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'
vid_dir = Path(PIPELINE_DIR) / 'outputs' / 'video'
videos  = sorted(vid_dir.glob('*.mp4'), key=lambda x: x.stat().st_mtime, reverse=True)

if not videos:
    print('No videos yet - run Cell 3 first')
else:
    items = []
    total_mb = 0
    for v in videos:
        mb = v.stat().st_size / 1024 / 1024
        total_mb += mb
        thumb = vid_dir / (v.stem + '_thumb.jpg')
        thumb_tag = f'<img src="{thumb}" width=200>' if thumb.exists() else ''
        items.append(
            f'<tr><td>{thumb_tag}</td>'
            f'<td><b>{v.name}</b><br>{mb:.1f} MB</td></tr>'
        )
        print(f'{v.name}  ({mb:.1f} MB)')

    print(f'\nTotal: {len(videos)} videos, {total_mb:.1f} MB')
    print('Download from the Output tab (right panel in Kaggle)')
    display(HTML('<h3>Generated Videos</h3><table>' + ''.join(items) + '</table>'))

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5 — BATCH MODE  (multiple niches, run overnight)
# ═══════════════════════════════════════════════════════════
import subprocess, sys, os, time
from pathlib import Path

PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'

BATCH_NICHES = [
    'personal_finance',
    'saas_tools',
    # 'legal_tax',
    # 'senior_health',
    # 'real_estate',
]

print(f'Batch: {len(BATCH_NICHES)} videos (~{len(BATCH_NICHES)*15} min total)')
print('=' * 60)

results = []
total_start = time.time()

for i, niche in enumerate(BATCH_NICHES, 1):
    print(f'\n[{i}/{len(BATCH_NICHES)}] {niche} ...')
    t0 = time.time()
    r = subprocess.run(
        [sys.executable, 'main.py',
         '--niche', niche,
         '--no-db', '--approve',
         '--log-format', 'console'],
        cwd=PIPELINE_DIR,
        env=os.environ.copy(),
        timeout=3600,
        capture_output=True, text=True
    )
    elapsed = time.time() - t0
    ok = r.returncode == 0
    results.append((niche, ok, elapsed))
    status = 'OK' if ok else 'FAILED'
    print(f'  -> {status}  ({elapsed:.0f}s / {elapsed/60:.1f} min)')
    if not ok:
        lines = (r.stdout + r.stderr).split('\n')
        print('\n'.join(lines[-10:]))

total_elapsed = time.time() - total_start
print(f'\n{"="*60}')
print(f'BATCH COMPLETE  ({total_elapsed/60:.1f} min total)')
for niche, ok, t in results:
    print(f'  {"OK  " if ok else "FAIL"}  {niche:<25}  {t/60:.1f} min')

vid_dir = Path(PIPELINE_DIR) / 'outputs' / 'video'
videos  = sorted(vid_dir.glob('*.mp4'), key=lambda x: x.stat().st_mtime, reverse=True)
print(f'\nTotal videos: {len(videos)}')
for v in videos:
    print(f'  {v.name}  ({v.stat().st_size/1024/1024:.1f} MB)')